# NextLearn — Bayesian Knowledge Tracing

**What this replaces.** NextLearn's learning path is driven by `graph.json`: a hand-written
map where a human decided "sous-acquis 1.1 unlocks 1.2". Every student follows the same map.
That map encodes an *assumption* about prerequisites, not a measurement of what any particular
student has actually mastered — the same weakness the risk-model label had before the OULAD work.

Bayesian Knowledge Tracing replaces the static map with a **live, per-student, per-skill
mastery probability** that updates after every attempt. Instead of "everyone advances 1.1 -> 1.2",
you get "*this* student is at 0.82 mastery on loops, 0.31 on pointers" — and the path is drawn
from that.

**BKT models one skill as a hidden two-state variable** (not-mastered / mastered) with four
parameters:

| Parameter | Meaning |
|---|---|
| `p_L0` | probability the skill is already mastered before the first attempt (prior knowledge) |
| `p_T`  | probability of learning it on each opportunity (transition) |
| `p_G`  | probability of a correct answer while *not* mastered (a lucky guess) |
| `p_S`  | probability of a wrong answer while mastered (a careless slip) |

**The honest blocker, stated up front.** BKT learns from a student's *sequence* of attempts on
a skill. NextLearn's quiz route (`web.ts:2701-2709`) does `$pull` then `$push` on `quizResults`
by `lessonKey`, so it keeps only the **latest** attempt per sous-acquis — there is no sequence to
trace. So BKT cannot yet be fit on live NextLearn data. This notebook does what the IRT and OULAD
notebooks did: **implement and validate the method on real public data (ASSISTments), then give
the exact deployment path.** Section 5 is that path — a small, un-backfillable schema change.

**What this notebook establishes, in order:**
1. The implementation is correct — recovering known parameters from simulated data.
2. It predicts real students' next answers — next-step AUC on held-out students.
3. It produces the artefact that matters: a per-student mastery curve that rises as they learn.

In [ ]:
!pip -q install numpy scipy pandas matplotlib scikit-learn

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time, os, urllib.request, warnings
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")
RANDOM_STATE = 42

SURFACE, INK, MUTED, LINE = "#fcfcfb", "#1c1c1a", "#6b6b66", "#dcdcd6"
CAT = ["#1a6fb5", "#d1621b", "#8f4a9c", "#3f7d3f"]   # CVD-validated, fixed order
SEQ_HUE = "#1a6fb5"
GOOD, BAD = "#3f7d3f", "#c23b22"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": LINE, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": LINE, "grid.linewidth": 0.6, "grid.alpha": 0.7,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "600",
    "figure.dpi": 120, "lines.linewidth": 2,
})
print("ready")

## 1. The model

Standard 4-parameter BKT, written in plain numpy/scipy — no `pyBKT` dependency, so it drops into
`ml/` behind the FastAPI service exactly like the IRT code, and every step stays inspectable.

The core is a **forward recurrence**: at each opportunity it predicts P(correct) from the current
mastery belief, then applies Bayes' rule to update that belief given the observed answer, then
applies the learning transition. The same recurrence both scores a sequence (for fitting) and
traces the mastery curve (for the dashboard).

In [ ]:
"""Bayesian Knowledge Tracing (standard 4-parameter BKT), from scratch in
numpy/scipy so it ports into ml/ with no new dependency.

A skill's mastery is a hidden 2-state variable (not-mastered / mastered).
Four parameters, all in [0,1]:

  p_L0  P(mastered before the first opportunity)      -- prior knowledge
  p_T   P(not-mastered -> mastered) per opportunity   -- learning rate
  p_G   P(correct | not mastered)                     -- guess
  p_S   P(incorrect | mastered)                       -- slip

The forward recurrence both scores a sequence and traces the student's mastery
belief over time.
"""
import numpy as np
from scipy.optimize import minimize

EPS = 1e-9


def _forward(params, obs, collect_pred=False):
    """Runs BKT over one 0/1 sequence.

    Returns the sequence log-likelihood. If collect_pred, also returns the list
    of predicted P(correct) computed BEFORE seeing each observation (the honest
    next-step prediction) and the traced mastery beliefs after each step.
    """
    p_L0, p_T, p_G, p_S = params
    L = p_L0
    ll = 0.0
    preds, mastery = [], []
    for o in obs:
        p_corr = L * (1 - p_S) + (1 - L) * p_G
        p_corr = min(max(p_corr, EPS), 1 - EPS)
        if collect_pred:
            preds.append(p_corr)
        ll += np.log(p_corr) if o == 1 else np.log(1 - p_corr)
        # Bayesian posterior on mastery given the observation. Use the SAME
        # clamped p_corr in the denominator so it can never divide by ~0.
        if o == 1:
            cond = L * (1 - p_S) / p_corr
        else:
            cond = L * p_S / (1 - p_corr)
        cond = min(max(cond, 0.0), 1.0)
        # learning transition for the next opportunity
        L = min(max(cond + (1 - cond) * p_T, EPS), 1 - EPS)
        if collect_pred:
            mastery.append(L)
    if collect_pred:
        return ll, preds, mastery
    return ll


def neg_loglik(params, sequences):
    return -sum(_forward(params, s) for s in sequences)


def fit_bkt(sequences, n_restarts=6, seed=0):
    """Fits the 4 BKT parameters by maximum likelihood over many student
    sequences for ONE skill. Bounds keep guess/slip < 0.5 (the standard
    identifiability constraint that rules out the degenerate 'model reversal').
    """
    rng = np.random.default_rng(seed)
    bounds = [(0.01, 0.99), (0.01, 0.99), (0.01, 0.49), (0.01, 0.49)]
    best = None
    for i in range(n_restarts):
        x0 = [rng.uniform(0.1, 0.9), rng.uniform(0.05, 0.4),
              rng.uniform(0.05, 0.4), rng.uniform(0.05, 0.4)] if i else [0.3, 0.1, 0.2, 0.1]
        res = minimize(neg_loglik, x0, args=(sequences,), method="L-BFGS-B", bounds=bounds)
        if best is None or res.fun < best.fun:
            best = res
    p_L0, p_T, p_G, p_S = best.x
    return {"p_L0": p_L0, "p_T": p_T, "p_G": p_G, "p_S": p_S, "nll": best.fun}


def predict_next(params, sequences):
    """For every step of every sequence, the P(correct) BKT would have predicted
    from only the prior steps. Returns (predictions, actuals) for AUC scoring."""
    p = (params["p_L0"], params["p_T"], params["p_G"], params["p_S"])
    preds, actuals = [], []
    for s in sequences:
        _, pr, _ = _forward(p, s, collect_pred=True)
        preds.extend(pr)
        actuals.extend(s)
    return np.array(preds), np.array(actuals)


def trace_mastery(params, obs):
    """Mastery belief after each opportunity — the learning curve for one student."""
    p = (params["p_L0"], params["p_T"], params["p_G"], params["p_S"])
    _, _, mastery = _forward(p, obs, collect_pred=True)
    return [params["p_L0"]] + mastery      # prepend the prior so the curve starts at L0


def simulate(params, n_students, seq_len, seed=0):
    """Generates sequences from the true generative BKT model, for recovery tests."""
    rng = np.random.default_rng(seed)
    p_L0, p_T, p_G, p_S = params
    out = []
    for _ in range(n_students):
        mastered = rng.random() < p_L0
        seq = []
        for _ in range(seq_len):
            correct = rng.random() < (1 - p_S) if mastered else rng.random() < p_G
            seq.append(int(correct))
            if not mastered and rng.random() < p_T:
                mastered = True
        out.append(seq)
    return out

## 2. Does the implementation work?

First the necessary check: generate sequences from known parameters, fit, and confirm recovery.
Nothing downstream is trustworthy if the fitter is biased.

In [ ]:
truth = {"p_L0": 0.25, "p_T": 0.15, "p_G": 0.20, "p_S": 0.10}
sim = simulate(tuple(truth.values()), n_students=800, seq_len=12, seed=RANDOM_STATE)
fit = fit_bkt(sim, seed=1)

print(f"{'param':<6}{'true':>8}{'recovered':>12}{'error':>9}")
for k in ["p_L0", "p_T", "p_G", "p_S"]:
    print(f"{k:<6}{truth[k]:>8.3f}{fit[k]:>12.3f}{abs(fit[k]-truth[k]):>9.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.6, 4.4))
names = ["p_L0", "p_T", "p_G", "p_S"]
x = np.arange(len(names))
ax.bar(x - 0.2, [truth[k] for k in names], width=0.38, color=MUTED, label="true")
ax.bar(x + 0.2, [fit[k] for k in names], width=0.38, color=SEQ_HUE, label="recovered")
ax.set_xticks(x); ax.set_xticklabels(names)
ax.set_ylabel("probability"); ax.set_title("Parameter recovery on simulated data")
ax.legend(frameon=False, fontsize=9); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

## 3. Real data — ASSISTments

ASSISTments has exactly what NextLearn is discarding: one row per student per question, with a
binary `correct` and an `order_id` that recovers the **temporal sequence** BKT needs. Only first
encounters are kept (`original == 1`), because BKT models genuine practice opportunities.

In [ ]:
if not os.path.exists("as.csv"):
    print("downloading ASSISTments (~83 MB) ...")
    urllib.request.urlretrieve("https://raw.githubusercontent.com/CAHLR/pyBKT-examples/master/data/as.csv", "as.csv")

d = pd.read_csv("as.csv", encoding="latin-1", low_memory=False,
                usecols=["order_id", "user_id", "skill_name", "correct", "original"])
d = d[d["original"] == 1].dropna(subset=["skill_name"])
d["correct"] = pd.to_numeric(d["correct"], errors="coerce")
d = d[d["correct"].isin([0, 1])]
d["correct"] = d["correct"].astype(int)
print(f"{len(d):,} responses | {d.user_id.nunique():,} students | {d.skill_name.nunique()} skills")
d.skill_name.value_counts().head(6)

### Fit per skill, and predict held-out students' next answers

BKT is fit **per skill** (each skill has its own learning dynamics). The honest evaluation is
**next-step prediction on students the model never saw**: split students 70/30, fit on the train
students' sequences, then for every step of every test student, predict their answer from only
their prior answers. Higher AUC = the mastery belief is tracking real ability.

In [ ]:
def seqs_for(skill, users):
    s = d[(d.skill_name == skill) & (d.user_id.isin(users))].sort_values("order_id")
    return [g["correct"].tolist() for _, g in s.groupby("user_id") if len(g) >= 1]

# A naive tracer: predict each step from the student's own success rate so far.
# If BKT can't beat this, its two-state hidden model is buying nothing.
def running_mean_baseline(sequences, prior=0.5):
    preds, actuals = [], []
    for s in sequences:
        c = n = 0
        for o in s:
            preds.append(prior if n == 0 else c / n)
            actuals.append(o); c += o; n += 1
    return np.array(preds), np.array(actuals)

SKILLS = ["Equation Solving Two or Fewer Steps", "Addition and Subtraction Integers",
          "Conversion of Fraction Decimals Percents", "Venn Diagram",
          "Volume Cylinder", "Percent Of"]

rng = np.random.default_rng(RANDOM_STATE)
rows, bkt_p, bkt_a, base_p, base_a = [], [], [], [], []
for skill in SKILLS:
    users = d[d.skill_name == skill].user_id.unique(); rng.shuffle(users)
    cut = int(len(users) * 0.7)
    tr, te = seqs_for(skill, users[:cut]), seqs_for(skill, users[cut:])
    if len(tr) < 30 or len(te) < 10:
        continue
    fit = fit_bkt(tr, seed=1)
    p, a = predict_next(fit, te)
    bp, ba = running_mean_baseline(te)
    auc = roc_auc_score(a, p) if len(set(a)) > 1 else np.nan
    bauc = roc_auc_score(ba, bp) if len(set(ba)) > 1 else np.nan
    bkt_p.extend(p); bkt_a.extend(a); base_p.extend(bp); base_a.extend(ba)
    rows.append({"skill": skill, "train": len(tr), "test": len(te),
                 "BKT AUC": auc, "baseline AUC": bauc,
                 "p_L0": fit["p_L0"], "p_T": fit["p_T"], "p_G": fit["p_G"], "p_S": fit["p_S"]})

res = pd.DataFrame(rows)
pooled_bkt = roc_auc_score(bkt_a, bkt_p)
pooled_base = roc_auc_score(base_a, base_p)
print(f"POOLED next-step AUC   BKT {pooled_bkt:.3f}   vs   running-mean baseline {pooled_base:.3f}")
res.round(3)

In [ ]:
plot = res.dropna(subset=["BKT AUC"]).sort_values("BKT AUC")
y = np.arange(len(plot))
fig, ax = plt.subplots(figsize=(9, 0.55 * len(plot) + 1.8))
ax.barh(y - 0.2, plot["baseline AUC"], height=0.38, color=MUTED, label="running-mean baseline")
ax.barh(y + 0.2, plot["BKT AUC"], height=0.38, color=SEQ_HUE, label="BKT")
ax.axvline(0.5, color=BAD, linestyle="--", linewidth=1.2)
ax.text(0.5, len(plot) - 0.4, " chance", color=BAD, fontsize=9)
ax.set_yticks(y); ax.set_yticklabels([s[:34] for s in plot["skill"]])
ax.set_xlim(0.45, 1.0); ax.set_xlabel("next-step prediction AUC (held-out students)")
ax.set_title("BKT vs a naive tracer, per skill"); ax.legend(frameon=False, fontsize=9)
ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

BKT matching or beating the running-mean baseline is the point: the two-state hidden model, with
its guess and slip parameters, extracts more signal than simply averaging a student's past answers.
The `p_G` / `p_S` values are also readable on their own — a skill with high `p_G` is one where
students can guess correctly without understanding, which is itself useful feedback about the items.

## 4. The artefact that matters — a single student's mastery curve

This is what the dashboard would show, and what makes BKT concrete: one student's mastery belief
on one skill, rising (or not) as they answer. Green = they got it right, red = wrong. The line is
BKT's live estimate of P(mastered) after each answer.

In [ ]:
SKILL = "Addition and Subtraction Integers"
fit = fit_bkt(seqs_for(SKILL, d[d.skill_name == SKILL].user_id.unique()), seed=1)

# Pick a real student with a clear learning trajectory: shaky start, then a run
# of success that actually carries BKT across the mastery threshold.
MASTERY = 0.95
cand = (d[d.skill_name == SKILL].sort_values("order_id")
          .groupby("user_id")["correct"].agg(list))
cand = cand[cand.map(len).between(6, 12)]
uid = next(u for u, s in cand.items()
           if s[0] == 0 and sum(s[-3:]) == 3 and max(trace_mastery(fit, s)) >= MASTERY)
obs = cand.loc[uid]
mastery = trace_mastery(fit, obs)

fig, ax = plt.subplots(figsize=(9, 4.6))
xs = np.arange(len(mastery))
ax.plot(xs, mastery, color=SEQ_HUE, marker="o", markersize=5, zorder=3, label="P(mastered)")
ax.axhline(MASTERY, color=MUTED, linestyle=":", linewidth=1)
ax.text(0, MASTERY + 0.01, f" mastery threshold ({MASTERY})", color=MUTED, fontsize=9)
for i, o in enumerate(obs):
    ax.scatter(i + 1, -0.06, marker="s", s=90, color=GOOD if o == 1 else BAD,
               clip_on=False, zorder=4)
ax.set_ylim(-0.1, 1.05); ax.set_xlim(-0.2, len(mastery) - 0.8)
ax.set_xlabel("opportunity  (squares: green = correct, red = wrong)")
ax.set_ylabel("BKT mastery belief")
ax.set_title(f"Student {uid} learning '{SKILL}'")
ax.legend(frameon=False, fontsize=9, loc="lower right")
plt.tight_layout(); plt.show()

reached = next((i for i, m in enumerate(mastery) if m >= 0.95), None)
print(f"answers: {obs}")
print(f"BKT declares mastery (>=0.95) after opportunity {reached}" if reached is not None
      else "BKT never reaches the 0.95 mastery threshold for this student")

Read the curve, not just the answers: BKT doesn't jump to "mastered" on the first correct answer,
because early success could be a guess (`p_G`). It requires a *run* of success before the belief
crosses the threshold — and a later slip dents it without erasing it. That is the behaviour a
prerequisite graph cannot express, and it is per-student.

## 5. What this would take in NextLearn

**This notebook changes nothing in the app.** Deploying BKT means, in order:

**1 — Capture the attempt sequence (the blocking change).** Today `web.ts:2701` deletes the prior
`quizResults` entry for a lesson before pushing the new one, so only the latest survives. BKT needs
the history. Append instead of replace, recording each attempt's binary outcome:

```ts
"progress.skillAttempts": {   // a NEW array, append-only
  subAcquisId, correct: passed, submittedAt: new Date()
}
```

This is the **same un-backfillable data** the IRT notebook needs — every attempt taken before this
change is lost to knowledge tracing. It is the one urgent step.

**2 — Fit periodically in Python.** Add a `/bkt/fit` endpoint to `shap_service.py` that fits the
4 parameters per sous-acquis on the accumulated sequences and writes `data/bkt-params.json`.
Parameters drift slowly, so this is a scheduled job, not a per-request call.

**3 — Trace mastery live.** A cheap `/bkt/mastery` call (or a pure-JS port of `trace_mastery`,
since inference is trivial) turns a student's attempt history into a mastery vector per skill.

**4 — Let mastery drive the path.** This is the payoff: instead of `graph.json` deciding 1.1 -> 1.2
for everyone, a sous-acquis unlocks when its prerequisites cross a mastery threshold *for that
student*. `graph.json` becomes the prerequisite structure; BKT supplies the per-student readiness.
It also feeds the **Agent Profiling** blackboard as the per-skill mastery component of the profile.

### The limitation to state plainly

BKT needs enough attempts per skill per student to trace a curve — one attempt gives you only the
prior. With ~35 students and one attempt per sous-acquis kept today, you cannot fit trustworthy
NextLearn parameters yet. What is defensible now is exactly this: a validated implementation,
demonstrated on 4,000+ real students, with the capture path defined so tracing becomes possible as
usage grows. Same posture as the OULAD and IRT work — the method is sound, the local numbers need
students.